# 01 · Basic RAG Pipeline

> **Module:** 04 – Retrieval-Augmented Generation  
> **Objective:** Build a complete RAG system from scratch, understanding every component and its engineering trade-offs.

---

## 🎯 Learning Objectives

1. Understand why RAG exists and when to use it vs fine-tuning
2. Build each component of a RAG pipeline independently
3. Understand chunking strategies and their effect on retrieval quality
4. Implement semantic search with a vector database
5. Evaluate RAG output quality with quantitative metrics

---

## 1. Why RAG?

LLMs have two fundamental limitations:

| Problem | Description | RAG Solution |
|---------|-------------|-------------|
| **Knowledge cutoff** | Model can't know events after training | Inject current documents at runtime |
| **Hallucination** | Model confidently generates false facts | Ground answers in retrieved evidence |
| **Private data** | Model wasn't trained on your documents | Retrieve from your own knowledge base |
| **Context limit** | Can't fit entire knowledge base in prompt | Retrieve only relevant chunks |

**RAG vs Fine-tuning:**
```
Fine-tune when:  knowledge is stable, needs deep understanding, style matters
RAG when:        knowledge changes, need citations, fast iteration, cost-sensitive
```

---

## 2. RAG Architecture

```
┌─────────────────────────────────────────────────────────────────────┐
│                         RAG PIPELINE                                 │
│                                                                       │
│  INDEXING (offline)              RETRIEVAL + GENERATION (online)      │
│  ─────────────────               ─────────────────────────────────    │
│                                                                       │
│  Documents                       User Query                          │
│      │                               │                               │
│      ▼                               ▼                               │
│  ┌─────────┐                   ┌───────────┐                         │
│  │  Chunk  │                   │  Embed    │                         │
│  │  Docs   │                   │  Query    │                         │
│  └────┬────┘                   └─────┬─────┘                        │
│       │                              │                               │
│       ▼                              ▼                               │
│  ┌─────────┐                   ┌───────────┐                         │
│  │  Embed  │                   │  Retrieve │                         │
│  │  Chunks │──→ Vector DB ←────│  Top-K    │                         │
│  └─────────┘                   └─────┬─────┘                        │
│                                      │                               │
│                                      ▼                               │
│                                ┌───────────┐                         │
│                                │  Augment  │ ← [context + query]    │
│                                └─────┬─────┘                        │
│                                      │                               │
│                                      ▼                               │
│                                ┌───────────┐                         │
│                                │    LLM    │                         │
│                                │ Generate  │                         │
│                                └─────┬─────┘                        │
│                                      │                               │
│                                      ▼                               │
│                                  Answer ✅                           │
└─────────────────────────────────────────────────────────────────────┘
```

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
!pip install sentence-transformers faiss-cpu openai anthropic tiktoken -q

import os
import re
import json
import time
import numpy as np
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple
from sentence_transformers import SentenceTransformer
import faiss

# Set your API key
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
# os.environ["OPENAI_API_KEY"] = "sk-..."

## 3. Component 1: Document Chunking

In [ ]:
@dataclass
class Chunk:
    """A piece of a document with metadata."""
    text: str
    doc_id: str
    chunk_index: int
    start_char: int
    end_char: int
    metadata: Dict = field(default_factory=dict)


class DocumentChunker:
    """
    Implements multiple chunking strategies.
    
    Chunking strategy dramatically affects retrieval quality:
    - Too small: chunks lack context, embeddings are noisy
    - Too large: chunks contain multiple topics, retrieval is imprecise
    - Overlap: reduces the chance of splitting relevant context
    """

    @staticmethod
    def fixed_size(
        text: str,
        doc_id: str,
        chunk_size: int = 512,
        overlap: int = 64
    ) -> List[Chunk]:
        """Split by character count with overlap."""
        chunks = []
        step = chunk_size - overlap
        for i, start in enumerate(range(0, len(text), step)):
            end = min(start + chunk_size, len(text))
            chunks.append(Chunk(
                text=text[start:end],
                doc_id=doc_id,
                chunk_index=i,
                start_char=start,
                end_char=end
            ))
            if end == len(text):
                break
        return chunks

    @staticmethod
    def sentence_based(
        text: str,
        doc_id: str,
        sentences_per_chunk: int = 5,
        overlap_sentences: int = 1
    ) -> List[Chunk]:
        """Split on sentence boundaries — preserves semantic units."""
        sentences = re.split(r'(?<=[.!?])\s+', text.strip())
        chunks = []
        step = sentences_per_chunk - overlap_sentences
        for i in range(0, len(sentences), step):
            chunk_sents = sentences[i: i + sentences_per_chunk]
            chunk_text = ' '.join(chunk_sents)
            chunks.append(Chunk(
                text=chunk_text,
                doc_id=doc_id,
                chunk_index=len(chunks),
                start_char=text.find(chunk_sents[0]) if chunk_sents else 0,
                end_char=0,
                metadata={"num_sentences": len(chunk_sents)}
            ))
        return chunks

    @staticmethod
    def semantic_sections(
        text: str,
        doc_id: str,
    ) -> List[Chunk]:
        """Split on markdown headers or blank lines — preserves document structure."""
        sections = re.split(r'\n#{1,3} |\n\n\n+', text)
        chunks = []
        for i, section in enumerate(sections):
            section = section.strip()
            if len(section) > 50:  # skip empty sections
                chunks.append(Chunk(
                    text=section,
                    doc_id=doc_id,
                    chunk_index=i,
                    start_char=text.find(section),
                    end_char=0
                ))
        return chunks


# Compare chunking strategies on the same document
sample_doc = """
Transformers revolutionized natural language processing. The attention mechanism 
allows the model to weigh relationships between all words in a sequence simultaneously. 
This parallelism made training dramatically faster than RNNs.

BERT introduced bidirectional pre-training. Unlike GPT which processes text left-to-right,
BERT reads the entire sequence at once. This enabled superior performance on classification
and question-answering tasks. BERT was pre-trained on BookCorpus and Wikipedia.

GPT-3 demonstrated that scale unlocks emergent capabilities. With 175 billion parameters,
it showed few-shot learning without any fine-tuning. The key insight was that a sufficiently
large language model learns to perform tasks from just a few examples in the prompt.
This led to the prompting paradigm that dominates modern LLM applications.
"""

chunker = DocumentChunker()
strategies = {
    "Fixed-size (512 chars)": chunker.fixed_size(sample_doc, "doc1", chunk_size=200, overlap=30),
    "Sentence-based (3 sent)": chunker.sentence_based(sample_doc, "doc1", sentences_per_chunk=3),
    "Semantic sections": chunker.semantic_sections(sample_doc, "doc1"),
}

for strategy_name, chunks in strategies.items():
    print(f"\n{'='*60}")
    print(f"Strategy: {strategy_name} → {len(chunks)} chunks")
    for c in chunks:
        print(f"  Chunk {c.chunk_index} ({len(c.text)} chars): {c.text[:80]}...")

## 4. Component 2: Embedding & Vector Index

In [ ]:
class VectorStore:
    """
    In-memory vector store using FAISS.
    
    In production you'd use: ChromaDB, Pinecone, Weaviate, Qdrant, pgvector.
    The principles are the same — this implementation makes them transparent.
    """

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        print(f"Loading embedding model: {model_name}")
        self.model = SentenceTransformer(model_name)
        self.dim = self.model.get_sentence_embedding_dimension()
        print(f"Embedding dimension: {self.dim}")

        # FAISS index — flat L2 (exact search, good for <100k docs)
        # For larger corpora use: faiss.IndexIVFFlat or faiss.IndexHNSWFlat
        self.index = faiss.IndexFlatIP(self.dim)  # Inner Product = cosine if normalized
        self.chunks: List[Chunk] = []

    def add_chunks(self, chunks: List[Chunk], batch_size: int = 64) -> None:
        """Embed and index chunks."""
        texts = [c.text for c in chunks]
        print(f"Embedding {len(texts)} chunks...")
        embeddings = self.model.encode(
            texts,
            batch_size=batch_size,
            normalize_embeddings=True,  # normalize for cosine similarity via IP
            show_progress_bar=True
        )
        self.index.add(embeddings.astype(np.float32))
        self.chunks.extend(chunks)
        print(f"Index now contains {self.index.ntotal} vectors")

    def search(
        self,
        query: str,
        top_k: int = 5,
        score_threshold: float = 0.0
    ) -> List[Tuple[Chunk, float]]:
        """Retrieve top-k most similar chunks."""
        query_embedding = self.model.encode(
            [query],
            normalize_embeddings=True
        ).astype(np.float32)

        scores, indices = self.index.search(query_embedding, top_k)

        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx >= 0 and score >= score_threshold:
                results.append((self.chunks[idx], float(score)))
        return results


# Build a small knowledge base
documents = [
    ("transformers", """
     The Transformer architecture was introduced in "Attention Is All You Need" (2017).
     It uses multi-head self-attention to process sequences in parallel.
     The encoder-decoder structure made it ideal for machine translation.
     Transformers replaced RNNs as the dominant sequence modeling architecture.
     """),
    ("bert", """
     BERT (Bidirectional Encoder Representations from Transformers) was published by Google in 2018.
     It uses masked language modeling and next sentence prediction as pre-training objectives.
     BERT achieved state-of-the-art on 11 NLP benchmarks upon release.
     The key innovation is bidirectional context — each word can attend to all others.
     """),
    ("gpt", """
     GPT (Generative Pre-trained Transformer) uses causal language modeling.
     GPT-3 has 175 billion parameters and was trained on 300 billion tokens.
     InstructGPT fine-tuned GPT-3 with RLHF to follow instructions.
     GPT-4 is multimodal and significantly more capable at reasoning tasks.
     """),
    ("rag", """
     RAG (Retrieval-Augmented Generation) combines retrieval with generation.
     Documents are chunked, embedded, and stored in a vector database.
     At query time, relevant chunks are retrieved and injected into the prompt.
     RAG reduces hallucinations and allows models to access up-to-date information.
     """)
]

chunker = DocumentChunker()
all_chunks = []
for doc_id, text in documents:
    chunks = chunker.sentence_based(text, doc_id, sentences_per_chunk=2)
    all_chunks.extend(chunks)

store = VectorStore("all-MiniLM-L6-v2")
store.add_chunks(all_chunks)

In [ ]:
# Test retrieval quality
queries = [
    "How does BERT differ from GPT?",
    "What is the attention mechanism?",
    "How does RAG reduce hallucinations?",
    "Who published BERT?",
]

for query in queries:
    print(f"\n🔍 Query: {query}")
    results = store.search(query, top_k=2)
    for i, (chunk, score) in enumerate(results):
        print(f"  [{i+1}] Score={score:.3f} | doc={chunk.doc_id}")
        print(f"       {chunk.text.strip()[:120]}...")

## 5. Component 3: Generation with Retrieved Context

In [ ]:
class RAGPipeline:
    """
    Complete RAG pipeline: retrieve relevant chunks → augment prompt → generate answer.
    
    This is the core pattern. Advanced RAG builds on this with:
    - Query rewriting
    - Reranking
    - Iterative retrieval
    - Hybrid search
    """

    SYSTEM_PROMPT = """You are a helpful assistant. Answer questions using ONLY the provided context.
If the context doesn't contain enough information, say "I don't have enough information to answer this."
Always cite which document your answer comes from."""

    def __init__(self, vector_store: VectorStore, llm_client=None, top_k: int = 3):
        self.store = vector_store
        self.llm = llm_client
        self.top_k = top_k

    def build_prompt(self, query: str, retrieved: List[Tuple[Chunk, float]]) -> str:
        """Construct the augmented prompt from query + context."""
        context_parts = []
        for i, (chunk, score) in enumerate(retrieved):
            context_parts.append(
                f"[Document {i+1} | source: {chunk.doc_id} | relevance: {score:.2f}]\n"
                f"{chunk.text.strip()}"
            )
        context = "\n\n".join(context_parts)

        return f"""Context:
---
{context}
---

Question: {query}

Answer:"""

    def query(
        self,
        question: str,
        verbose: bool = True
    ) -> Dict:
        """End-to-end RAG: retrieve → augment → generate."""
        t0 = time.perf_counter()

        # Step 1: Retrieve
        retrieved = self.store.search(question, top_k=self.top_k)
        retrieval_time = time.perf_counter() - t0

        # Step 2: Build prompt
        prompt = self.build_prompt(question, retrieved)

        if verbose:
            print(f"\n{'='*60}")
            print(f"Question: {question}")
            print(f"Retrieved {len(retrieved)} chunks in {retrieval_time*1000:.1f}ms")
            for chunk, score in retrieved:
                print(f"  → [{chunk.doc_id}] score={score:.3f}: {chunk.text.strip()[:60]}...")
            print(f"\nAugmented Prompt:\n{prompt}")

        # Step 3: Generate (using real API if available, otherwise mock)
        if self.llm:
            import anthropic
            t1 = time.perf_counter()
            response = self.llm.messages.create(
                model="claude-3-haiku-20240307",
                max_tokens=512,
                system=self.SYSTEM_PROMPT,
                messages=[{"role": "user", "content": prompt}]
            )
            answer = response.content[0].text
            generation_time = time.perf_counter() - t1
        else:
            # Mock response for demonstration
            answer = f"[Mock Answer] Based on the retrieved context from {[c.doc_id for c, _ in retrieved]}: ..."
            generation_time = 0.0

        return {
            "question": question,
            "answer": answer,
            "sources": [(c.doc_id, round(s, 3)) for c, s in retrieved],
            "retrieval_ms": round(retrieval_time * 1000, 1),
            "generation_ms": round(generation_time * 1000, 1),
        }


# Initialize (pass your anthropic client for real answers)
# import anthropic
# client = anthropic.Anthropic()
rag = RAGPipeline(store, llm_client=None, top_k=3)

result = rag.query("What is BERT and how does it differ from GPT?", verbose=True)
print(f"\nAnswer: {result['answer']}")
print(f"Sources: {result['sources']}")
print(f"Retrieval: {result['retrieval_ms']}ms")

## 6. Experiment: Chunking Strategy vs Retrieval Quality

In [ ]:
import matplotlib.pyplot as plt

# Ground truth QA pairs for evaluation
eval_pairs = [
    {"question": "What pre-training objectives does BERT use?",
     "expected_doc": "bert"},
    {"question": "How many parameters does GPT-3 have?",
     "expected_doc": "gpt"},
    {"question": "What problem does RAG solve?",
     "expected_doc": "rag"},
    {"question": "What did Attention Is All You Need introduce?",
     "expected_doc": "transformers"},
    {"question": "What is masked language modeling?",
     "expected_doc": "bert"},
]

def evaluate_retrieval(store: VectorStore, eval_pairs: list, top_k: int = 3) -> dict:
    """Measure retrieval precision@k."""
    hits = 0
    for pair in eval_pairs:
        results = store.search(pair["question"], top_k=top_k)
        retrieved_docs = [c.doc_id for c, _ in results]
        if pair["expected_doc"] in retrieved_docs:
            hits += 1
    precision = hits / len(eval_pairs)
    return {"precision_at_k": precision, "hits": hits, "total": len(eval_pairs)}

# Compare chunk sizes
chunk_configs = [
    ("tiny (100 chars)",   100, 20),
    ("small (200 chars)",  200, 40),
    ("medium (400 chars)", 400, 80),
    ("large (600 chars)",  600, 100),
]

results_by_config = {}
for config_name, chunk_size, overlap in chunk_configs:
    chunks_for_eval = []
    for doc_id, text in documents:
        chunks_for_eval.extend(chunker.fixed_size(text, doc_id, chunk_size, overlap))
    
    test_store = VectorStore("all-MiniLM-L6-v2")
    test_store.add_chunks(chunks_for_eval)
    
    eval_result = evaluate_retrieval(test_store, eval_pairs)
    results_by_config[config_name] = eval_result
    print(f"{config_name:<25} P@3={eval_result['precision_at_k']:.2f} "
          f"({eval_result['hits']}/{eval_result['total']}) | chunks={len(chunks_for_eval)}")

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
names = list(results_by_config.keys())
scores = [v['precision_at_k'] for v in results_by_config.values()]
bars = ax.bar(names, scores, color=['tomato', 'orange', 'steelblue', 'seagreen'])
ax.bar_label(bars, fmt='{:.2f}', padding=3)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Precision@3')
ax.set_title('Retrieval Precision vs Chunk Size')
ax.axhline(y=0.8, color='gray', linestyle='--', alpha=0.5, label='Target P@3 = 0.8')
ax.legend()
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('../figures/04_chunking_vs_retrieval.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Engineering Notes

### Embedding Model Selection

| Model | Dim | Speed | Quality | Use Case |
|-------|-----|-------|---------|----------|
| `all-MiniLM-L6-v2` | 384 | ⚡⚡⚡ | Good | Prototyping, low latency |
| `all-mpnet-base-v2` | 768 | ⚡⚡ | Better | General purpose |
| `text-embedding-3-small` | 1536 | ⚡ | Great | Production (OpenAI) |
| `text-embedding-3-large` | 3072 | slow | Best | High-accuracy retrieval |
| `bge-large-en-v1.5` | 1024 | ⚡⚡ | Great | Open-source production |

### Vector Index Selection

| Index | Search | Memory | When to Use |
|-------|--------|--------|-------------|
| `IndexFlatIP` | Exact | High | < 100K docs, accuracy required |
| `IndexIVFFlat` | Approx | Medium | 100K–10M docs |
| `IndexHNSWFlat` | Approx | Medium | Low latency requirement |
| `IndexIVFPQ` | Approx | Low | Billions of vectors |

### ⚠️ Top RAG Failure Modes

1. **Bad chunking** — chunks split mid-sentence or mid-concept
2. **Wrong embedding model** — model not suited for domain (code vs prose)
3. **Missing context** — retrieved chunk doesn't have surrounding context
4. **Stale index** — index not updated when docs change
5. **Prompt too long** — too many retrieved chunks overwhelm the context window

## 8. Exercises

1. **Overlap Experiment**: Test overlap values of 0, 10%, 20%, 30% of chunk size. Plot P@3 vs overlap.

2. **Hybrid Search**: Combine dense retrieval (embedding similarity) with BM25 (keyword match) using reciprocal rank fusion. Compare to dense-only.

3. **Parent-Child Chunking**: Index small child chunks for retrieval, but return larger parent chunks for context. Implement and measure the quality difference.

4. **Query Rewriting**: Before retrieval, use an LLM to rewrite the user query into a better search query. Does this improve P@3?

5. **Full Pipeline**: Build a RAG system over 10 Wikipedia articles of your choice. Evaluate 20 questions and compute P@1, P@3, P@5.

---

## 9. References

- [Lewis et al. (2020) — Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks](https://arxiv.org/abs/2005.11401)
- [FAISS — A Library for Efficient Similarity Search](https://faiss.ai/)
- [MTEB Leaderboard — Embedding Model Rankings](https://huggingface.co/spaces/mteb/leaderboard)
- [Chunking Strategies for LLM Applications — Pinecone Blog](https://www.pinecone.io/learn/chunking-strategies/)